# R_P00 -- Prepare the panel (data preparation, R)

Reads every export under ROOT (default D:/LKT/RWDR/data; the old D:/LKT/TST_ArtalR until MIGRATE_DATA.bat has moved the data), builds the pixel panel with the SAME rules as the Python P00, averages the benchmark (BM) sites of each sub-watershed, codes every row for the fragment rule and reads your fund workbook. v20.57: it sets NO design -- each model notebook sets and applies its own (the options set to "data" are chosen FROM THE DATA, never from the effect).

In [ ]:
# the bundle's R folder (works from rstudio/, jupyter/ or the project root)
R_HOME_DIR <- normalizePath(if (file.exists("lib/reward_paths.R")) "." else "..", winslash = "/")
for (f in c("reward_paths.R", "reward_design.R", "reward_prep.R", "reward_models_core.R")) source(file.path(R_HOME_DIR, "lib", f), local = environment())
cat("data root:", ROOT, "| output:", OUTPUT_DIR, "\n")
invisible(confirm_packages())   # v20.55: are ALL R packages installed? (pre-built first; a missing one is installed now when AUTO_INSTALL_PACKAGES)

**The design is NOT set here (v20.57).** R_P00 builds the panel ONCE with every row -- all rings, years and seasons, every sub-watershed and every fragment of another sub-watershed, each row coded (`fragment`) -- and never has to be re-run for a design choice. The timing (your fund workbook by default), the dose, the control rings, the years, the seasons, OVERLAP_ROWS, FRAGMENT_RULE and POOLED_FE are set in the settings cell of EACH MODEL notebook (R_M01 ... R_M45, R_RUN_ALL_MODELS, R_D01) and applied when that model runs. The defaults live in `lib/reward_paths.R`. Change `ROOT` there if your exports are elsewhere.

In [ ]:
# v20.57: nothing to set here for the design -- it is set in each MODEL notebook (defaults: lib/reward_paths.R).
# v20.59 (YOUR RULE): PERIOD_RULE (lib/reward_paths.R) sets the panel's post / pre -- "treat" = the exports' Treat column (1 = post, 0 = pre),
# "year" = Year >= TREATMENT_YEAR, "both" = the two must agree (a disagreeing row leaves). buff_km 0 = the treatment area, 1-5 = the control
# rings. R_P00 confirms both on every input file -> input_design_audit_R.csv.
# The panel-building rules (pixel linkage, duplicates, masked zeros, negative covariates) stay in lib/reward_paths.R / reward_prep.R.
# v20.59 -- EVERY customisation at the panel level too (as Python's P00_Settings): these are the DEFAULTS the design report, the screen table
# and a model that leaves an option unset use; each model notebook sets ITS OWN values in its first chunk and they win for that model.
# The panel itself is built once with every row and never has to be re-run for a design choice (lib/reward_paths.R has the same defaults).
DESIGN_MODE       <- "recommended"   # "recommended": an option set to "data" is chosen FROM THE DATA (DESIGN_RECOMMENDATION.md) | "manual"
TREATMENT_TIMING  <- "fund"          # "fund" (the fund workbook: each sub-watershed's first treated season) | "registry" (sites.csv) | "fixed"
TREATMENT_YEAR    <- 2022            # "fixed" timing, and the fall-back for a sub-watershed the fund file / registry does not date
FUND_START_RULE   <- "backcast"      # "backcast" | "share" | "file_start"
DOSE_VARIABLE     <- "dose_intensity_per_ha"   # | "dose_amount_sws" | "dose_share_of_target"
CONTROL_RINGS     <- "data"          # "data" | 1:5 | 1:3 | c(2, 4)   (the TREATMENT AREA is buff_km 0)
PRE_YEARS         <- "data"          # "data" | NA (every year before the start) | a number of years | 2015 (a calendar year = the FIRST pre year)
POST_YEARS        <- "data"          # "data" | NA (every year from the start) | a number of years | 2025 (a calendar year = the LAST post year)
SEASONS           <- "all"           # "all" | "seasonal" | "yearly" | "Rabi" | "Kharif+Rabi"
EXCLUDE_TRANSITION_YEAR <- FALSE     # TRUE = the first treated year of each series leaves the sample
UNIT_FE           <- "pixel_season"  # | "pixel"
COHORT_OFFSET     <- 0L
POOLED_FE         <- "site_period"   # several sub-watersheds: sub-watershed x year x season effects | "period"
OVERLAP_ROWS      <- "drop"          # | "keep"
FRAGMENT_RULE     <- "drop"          # | "keep"  (fragments of other sub-watersheds inside the exports, minor sub-watersheds of the panel)
COVARIATES        <- c("Rain", "Tmax", "Tmean", "Tmin")
# v20.59 -- YOUR RULE: the sub-watershed of every row is where its latitude / longitude falls in the shapefile (the core or a control ring);
#   the file's id only labels. ONE sub-watershed processed = the one holding the majority of the rows (SUB_WATERSHEDS = "data": every
#   sub-watershed with >= FRAGMENT_MIN_SHARE of the largest one's own rows; the rest are fragments); SEVERAL = every row in its own sub-watershed.
SUB_WATERSHEDS      <- "data"        # "data" | "all" | 11 | c(7, 11)
SITE_GEOMETRY_CHECK <- TRUE          # every row located in the shapefile: confirmed / corrected / assigned | FALSE = trust the id in the file
BUFF_FROM_GEOMETRY  <- FALSE         # TRUE = buff_km always from the polygon ring | FALSE = only where the sub-watershed was corrected / assigned
PIXEL_ONE_SITE      <- TRUE          # v20.59 -- YOUR RULE: one sub-watershed and one ring per pixel in the whole panel (the polygon that holds the point decides;
                                     #   confirmed: panel_pixel_consistency_R.csv) | FALSE = the v20.58 rule (a point in two zones kept once per sub-watershed)
# v20.59 -- YOUR RULE: the panel KEEPS every row and every value. Fill values (a year-season with ONE value for every pixel) and gap-filled
# rows (GapFilled = 1 / Coverage 0) stay IN the panel; whether a MODEL estimates on them is decided by the two options below -- the DEFAULTS
# here (the design report and the screen table of this notebook use them) and, for each model, ITS OWN setting in its notebook (as Python's
# P00_Settings and every model's CELL 1). Keeping a fill year-season is not neutral: in it the treated-control difference is exactly 0,
# which dilutes the pre (or post) gap the DiD compares -- the screen's report says so when "keep" is in force.
OUTCOME_SCREEN    <- "drop"   # "drop" = a year-season constant across pixels (a fill value) or with collapsed coverage leaves the model (evidence:
                              #   OUTCOME_SCREEN_<outcome>.csv) | "keep" = reported and KEPT (results tagged _screenKept) | "off" = no screen
EXCLUDE_GAPFILLED <- TRUE     # TRUE = rows the exporter filled from history (GapFilled = 1 / Coverage 0) are not observations | FALSE keeps them
DESIGN_SOURCE     <- "panel"  # v20.59 -- YOUR RULE: the PANEL's treat / control / pre / post / did (the exports' Treat flag, PERIOD_RULE) are what every
                              #   model estimates on by default (results tagged _panelDesign; a notebook's CELL 1 overrides it) | "model" = design-based
# ================================================================================================================================
# OPTIONAL CUSTOMISATIONS (1 Oct, your rule) -- every one has a USE_ switch. FALSE = NOT applied at all, whatever the value beside it
# says (the standard design above runs as it is); TRUE = applied exactly as set. All are FALSE by default. DESIGN IN EFFECT shows each
# one as "not used" or with the value in force; a rule in force also tags the results folder (the tag is named beside each option).
# HERE: the panel-level DEFAULTS of these switches and values (a model notebook's own SECTION B / C wins for that model).
# --------------------------------------------------------------------------------------------------------------------------------
# SECTION B -- THE CONTROL GROUP CHOSEN ON THE PRE PERIOD (your request of 30 Sep): which buffers / clusters are the controls for THIS
# outcome, decided on the pre period only and then FIXED across the whole panel (the same control pixels in every year and season)
# --------------------------------------------------------------------------------------------------------------------------------
USE_CONTROL_SELECTION <- FALSE    # FALSE = every ring of CONTROL_RINGS is the control group (as above) | TRUE = CONTROL_SELECTION below chooses it
CONTROL_SELECTION <- "pre_rings"  #   "pre_rings": the CONTROL_SELECT_K (1 or 2) buffers whose PRE-period series is closest to the treatment area's -- your
                                  #   "most close 1 or 2 buffers" | "pre_blocks": control CLUSTERS (~1 km blocks of pixels) from ANY part of the buffers, the
                                  #   closest first, until CONTROL_SELECT_RATIO x the treated pixels -- your "clusters from any part of the buffers".
                                  #   Per outcome; the decision uses the PRE period only (a post- or outcome-based rule is refused); made once and applied
                                  #   to every year and season. Evidence: CONTROL_SELECTION_<outcome>_R.csv beside the results. Tag _ctrlPre2r / _ctrlPreBlk3x
CONTROL_SELECT_K  <- 2L           #   pre_rings: how many buffers -- 1 or 2 (up to 5)
CONTROL_SELECT_RATIO <- 3         #   pre_blocks: control pixels >= this x the treated pixels
CONTROL_SELECT_ON <- "trend"      #   what "closest" means: "trend" = the demeaned pre series' distance (what the parallel-trends assumption asks; recommended)
                                  #   | "level" = the nearest pre-period MEAN (your "mean of the treatment area vs the buffer's mean") | "rmse" = the RMSE of the
                                  #   cell-wise pre differences | "both" = trend + level.  Tag suffix: none / L / R / B
USE_SAME_PIXELS   <- FALSE        # FALSE = a pixel may contribute to one side only (the v20.58 sample) | TRUE = the treated and control groups are the SAME
SAME_PIXELS       <- "pre_post"   #   pixels across the panel: "pre_post" = every pixel observed in pre AND post, else it leaves (tag _pixPP) | "all" = in
                                  #   every year-season of the sample (a balanced pixel set; tag _pixAll). The sample-integrity line confirms it at every run
# --------------------------------------------------------------------------------------------------------------------------------
# SECTION C -- PANEL-PREPARATION ENHANCEMENTS (the three specifications of 30 Sep); each with its own switch
# --------------------------------------------------------------------------------------------------------------------------------
USE_DONUT         <- FALSE        # FALSE = ring 1 is a control like the others | TRUE = DONUT_RINGS leave the control pool -- the spillover buffer next to the core
DONUT_RINGS       <- 1L           #   the ring(s) left out, e.g. 1L or c(1L, 2L); applied before any control choice. Tag _donut1
USE_LANDUSE_MASK  <- FALSE        # FALSE = every land-use class | TRUE = only pixels whose PRE-period (baseline) LandUse class is in LANDUSE_KEEP stay. Tag _lu2
LANDUSE_KEEP      <- 2L           #   your exporter's class code(s) for agriculture (a pixel's class is read on the pre period, so the works cannot move it)
USE_BASELINE_NDVI_MASK <- FALSE   # FALSE = no mask | TRUE = only pixels whose PRE-period mean NDVI exceeds BASELINE_NDVI_MIN stay (an agricultural mask). Tag _ndviPre0.25
BASELINE_NDVI_MIN <- 0.25         #   the threshold (an NDVI value)
USE_COVERAGE_THRESHOLD <- FALSE   # FALSE = the standard screen (a year-season below 5 % of the typical coverage leaves) | TRUE = below MIN_PIXEL_COVERAGE_PCT. Tag _cov70
MIN_PIXEL_COVERAGE_PCT <- 0.70    #   the share of the typical treated / control coverage a year-season must reach (your log: 2023 Zaid, 2025 Kharif / Zaid collapsed)
USE_DROP_SINGLETONS <- FALSE      # FALSE = series seen once stay | TRUE = they leave before the demeaning (the pre-flight). Tag _noSingle
USE_PRECISION_TOLERANCE <- FALSE  # FALSE = the no-data zero is an EXACT 0 (the v20.58 rule) | TRUE = |value| <= PRECISION_TOLERANCE is the no-data zero and a
PRECISION_TOLERANCE <- 1e-6       #   year-season is "constant across pixels" within it -- indices in [-1, 1] are never compared with exact equality
# The Rabi-only run of the specifications is SEASONS <- "Rabi" above; the ~1 km block clusters are CLUSTER <- "block" above.
# ================================================================================================================================
CLUSTER           <- "auto"   # "auto" (the sub-watersheds; fewer than 6 -> the years) | "block" (~1 km spatial blocks: many clusters, spatial correlation absorbed)
cat("design defaults (a model notebook overrides each):", DESIGN_MODE, "| timing", TREATMENT_TIMING, "| period rule (R_P00)", PERIOD_RULE, "| outcome screen", OUTCOME_SCREEN, "| design source", DESIGN_SOURCE, "| control selection", CONTROL_SELECTION, "| same pixels", SAME_PIXELS, "| donut", paste(DONUT_RINGS, collapse = ","), "| land use", paste(LANDUSE_KEEP, collapse = ","), "| cluster", CLUSTER, "| gap-filled rows", if (isTRUE(EXCLUDE_GAPFILLED)) "left out" else "kept", "| rings", paste(CONTROL_RINGS, collapse = ","),
    "| seasons", paste(SEASONS, collapse = ","), "| fragments", FRAGMENT_RULE, "| overlap", OVERLAP_ROWS, "| pooled FE", POOLED_FE, "\n")


**Build the panel** -- exports, harmonised columns, masked zeros as missing, the sub-watershed from the shapefile (confirmed or corrected), every row coded for the fragment rule per export file, shifted grids linked, repeated rows dropped whole (the newer export's row kept as it is -- v20.58; DEDUP_FILL_FROM_DUPLICATES in lib/reward_prep.R), the BM means; the fund workbook's timing and dose tables (results/FUND) for you to read -- the models apply them when they run.

In [ ]:
FORCE_REBUILD <- FALSE   # 3 Oct: a valid panel on disk is KEPT (panel_is_valid_R: every required column, rows, Parquet DOUBLE, panel_build_settings_R.csv beside it);
                         #   TRUE = build it again. Every step of the build is said before it starts and when it ends (a long silent step is still running, not stuck)
panel <- if (!isTRUE(FORCE_REBUILD) && panel_is_valid_R()) { ok(sprintf("a valid panel is already there: %s (FORCE_REBUILD <- TRUE rebuilds it)", PANEL_PATH)); panel_precision_report_R(verbose = TRUE); invisible(NULL) } else run_prep()

**The design the DEFAULTS give** (export breaks, fill years, spillover into inner rings; the fund timing per sub-watershed) -- what a model uses when its settings are left as in lib/reward_paths.R. Each model applies its own settings when it runs.

In [ ]:
design <- prepare_design(); str(design[setdiff(names(design), c("choices", "notes"))])


**The outcome screen** -- which outcome-years are fill values (not data), per outcome.

In [ ]:
scr <- outcome_screen_R(OUTCOMES, design)   # v20.58: every outcome's usable years -- all rows at once, or out of core beyond 98 % of the RAM
fwrite(scr, file.path(RESULTS_DIR, "OUTCOME_SCREEN_R.csv")); print(scr)